# Stock Market Scanners

## How to use
| Step | Cell | When to run |
|------|------|-------------|
| 1 | **Install** | Once per session |
| 2 | **SGX Download** | Once — saves data to CSV |
| 3 | **SGX Scanner** | Anytime — loads from memory or CSV |
| 4 | **S&P 500 Download** | Once — saves data to CSV |
| 5 | **S&P 500 Scanner** | Anytime — loads from memory or CSV |
| 6 | **S&P 500 RSI Scanner** | Anytime — loads from memory or CSV |

> **Tip:** After running a Download cell, all scanner cells below it  
> will reuse the data already in memory — no re-download needed.

## Cell 1 — Install
Run once per Colab session.

In [ ]:
!pip install yfinance requests lxml -q

---
## Cell 2 — SGX Download
Downloads all SGX stocks + STI benchmark. Saves to `/content/`. Run once, then use the scanner without re-downloading.

In [ ]:
# ============================================================
# SGX DOWNLOAD — saves sgx_historical.csv + sti_close.csv
# Run once per session. Scanner cells load from these files.
# ============================================================
import csv, time, warnings
import requests, pandas as pd, yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

SGX_HIST_CSV = "/content/sgx_historical.csv"
STI_CSV      = "/content/sti_close.csv"
BATCH_SIZE   = 50
BENCHMARK    = "^STI"

print("=" * 55)
print("  SGX DATA DOWNLOAD")
print("=" * 55)

# ── Fetch SGX stock list ─────────────────────────────────────
print("\n[1/3] Fetching SGX stock list...")
resp = requests.get("https://api.sgx.com/securities/v1.1/",
                    headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
resp.raise_for_status()
stocks = [s for s in resp.json()["data"]["prices"] if s.get("type") == "stocks"]
sgx_ticker_info = {
    f"{s['nc']}.SI": {
        "symbol": s["nc"], "name": s.get("n", ""), "market": s.get("m", ""),
        "sgx_pe": s.get("pe"), "sgx_pb": s.get("pb"),
        "sgx_yld": s.get("yld") or s.get("dy"),
        "sgx_eps": s.get("eps") or s.get("es"),
        "sgx_mc":  s.get("mc")  or s.get("mktcap"),
    }
    for s in stocks if s.get("nc")
}
print(f"    Found {len(sgx_ticker_info)} stocks")

# ── Download price history ───────────────────────────────────
end_date   = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
print(f"\n[2/3] Downloading data ({start_date} → {end_date})...")

# STI benchmark
print(f"    Fetching benchmark {BENCHMARK}...")
sti_raw   = yf.download(BENCHMARK, start=start_date, end=end_date, auto_adjust=True, progress=False)
sti_close = sti_raw["Close"].squeeze()
sti_close.to_csv(STI_CSV)

all_tickers = list(sgx_ticker_info.keys())
batches     = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
sgx_stock_data = {}

with open(SGX_HIST_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["symbol", "name", "market", "date", "open", "high", "low", "close", "volume"])
    for i, batch in enumerate(batches):
        print(f"    Batch {i+1}/{len(batches)}...", end=" ", flush=True)
        try:
            raw = yf.download(batch, start=start_date, end=end_date,
                              auto_adjust=True, progress=False, threads=True)
            results = {}
            if len(batch) == 1:
                if not raw.empty:
                    results[batch[0]] = raw.rename(columns=str.lower)
            else:
                for t in batch:
                    try:
                        df = raw.xs(t, level=1, axis=1).dropna(how="all")
                        if not df.empty:
                            results[t] = df.rename(columns=str.lower)
                    except KeyError:
                        pass
            for yf_t, df in results.items():
                info = sgx_ticker_info[yf_t]
                sgx_stock_data[yf_t] = df
                for date, row in df.iterrows():
                    writer.writerow([
                        info["symbol"], info["name"], info["market"],
                        date.strftime("%Y-%m-%d"),
                        round(float(row.get("open",  0) or 0), 4),
                        round(float(row.get("high",  0) or 0), 4),
                        round(float(row.get("low",   0) or 0), 4),
                        round(float(row.get("close", 0) or 0), 4),
                        int(row.get("volume", 0) or 0),
                    ])
            print(f"ok ({len(results)}/{len(batch)})")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.3)

print(f"\n[3/3] Done!")
print(f"    Stocks downloaded : {len(sgx_stock_data)}")
print(f"    Saved to          : {SGX_HIST_CSV}")
print(f"    STI benchmark     : {STI_CSV}")
print("    ✅ Scanner cells can now use this data without re-downloading.")

## Cell 3 — SGX Stock Scanner
Scans SGX stocks for SMA Gap, Breakout, and OBV signals.  
**Loads from memory → CSV → downloads fresh** (no re-download if Cell 2 was run).

In [ ]:
# ============================================================
# SGX STOCK SCANNER
# Loads data from: memory → sgx_historical.csv → fresh download
# Filters: volume > 100,000 AND price > S$0.20
# ============================================================
import csv, os, time, warnings
import numpy as np, pandas as pd, requests, yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

SGX_HIST_CSV     = "/content/sgx_historical.csv"
STI_CSV          = "/content/sti_close.csv"
OUTPUT_RESULTS   = "/content/sgx_scan_results.csv"
OUTPUT_WATCHLIST = "/content/sgx_watchlist.txt"
BATCH_SIZE       = 50
MIN_VOLUME, MIN_PRICE = 100_000, 0.20


# ════════════════════════════════════════════════════════════
# SCANNER FUNCTIONS
# ════════════════════════════════════════════════════════════
def _sma(series, length):
    return series.rolling(length, min_periods=length).mean()

def add_sma_gap_rs_buy_sell_signals(df, index_close,
        b_sma20_len=20, b_sma50_len=50, b_slope_lookback=5,
        b_slope_smooth_len=1, a_rs_sma_length=30, a_delta_ma_length=9):
    out = df.copy()
    index_close = index_close.reindex(out.index).astype(float)
    out["b_sma20"]       = _sma(out["close"], b_sma20_len)
    out["b_sma50"]       = _sma(out["close"], b_sma50_len)
    out["b_raw_slope20"] = (out["b_sma20"] - out["b_sma20"].shift(b_slope_lookback)) / out["b_sma20"].shift(b_slope_lookback) * 100.0
    out["b_raw_slope50"] = (out["b_sma50"] - out["b_sma50"].shift(b_slope_lookback)) / out["b_sma50"].shift(b_slope_lookback) * 100.0
    out["b_slope20"]     = _sma(out["b_raw_slope20"], b_slope_smooth_len)
    out["b_slope50"]     = _sma(out["b_raw_slope50"], b_slope_smooth_len)
    out["b_bull"]        = out["b_slope20"] > out["b_slope50"]
    stock_sma        = _sma(out["close"], a_rs_sma_length)
    index_sma        = _sma(index_close, a_rs_sma_length)
    out["delta_ab"]  = (out["close"] / stock_sma) - (index_close / index_sma)
    out["delta_ma"]  = _sma(out["delta_ab"], a_delta_ma_length)
    out["slope_gap"] = out["b_slope20"] - out["b_slope50"]
    out["a_two_day_green_rising"] = (
        (out["delta_ab"] >= 0) & (out["delta_ab"].shift(1) >= 0) &
        (out["delta_ab"] > out["delta_ab"].shift(1)))
    out["green_candle"]       = out["close"] > out["open"]
    out["red_candle"]         = out["close"] < out["open"]
    out["slope_gap_reducing"] = (out["b_slope20"] > out["b_slope50"]) & (out["slope_gap"] < out["slope_gap"].shift(1))
    out["buy_condition"]      = out["b_bull"] & out["a_two_day_green_rising"] & out["green_candle"]
    out["sell_condition"]     = (out["delta_ab"] < out["delta_ma"]) & out["slope_gap_reducing"] & out["red_candle"]
    buy_signal = np.zeros(len(out), dtype=bool)
    sell_signal = np.zeros(len(out), dtype=bool)
    in_position = buy_ban = sell_ban = False
    for i in range(len(out)):
        buy  = bool(out["buy_condition"].iat[i])  and not buy_ban  and not in_position
        sell = bool(out["sell_condition"].iat[i]) and not sell_ban and in_position
        if buy:
            buy_signal[i] = True;  in_position = True;  buy_ban = True;  sell_ban = False
        if sell:
            sell_signal[i] = True; in_position = False; sell_ban = True; buy_ban = False
    out["buy_signal"] = buy_signal; out["sell_signal"] = sell_signal
    return out

def add_breakout_buy_signals(df, lookback_n1=30, lookback_n2=60, lookback_n3=90,
        sma200_slope_lookback=5, max_upside_pct=5.0):
    out = df.copy()
    out["sma20"]  = _sma(out["close"], 20)
    out["sma50"]  = _sma(out["close"], 50)
    out["sma200"] = _sma(out["close"], 200)
    out["sma200_slope_positive"] = out["sma200"] > out["sma200"].shift(sma200_slope_lookback)
    out["green_candle"]  = out["close"] > out["open"]
    out["sma20_above50"] = out["sma20"] > out["sma50"]
    out["b_above20sma"]  = out["close"] > out["sma20"]
    for label, lookback in [("n1", lookback_n1), ("n2", lookback_n2), ("n3", lookback_n3)]:
        signals = []
        for pos in range(len(out)):
            if pos < 1: signals.append(False); continue
            end_i = min(lookback, pos)
            a_price, a_offset = float("nan"), None
            for off in range(1, end_i + 1):
                h = out["high"].iat[pos - off]
                if pd.notna(h) and (pd.isna(a_price) or h > a_price):
                    a_price, a_offset = h, off
            pb = False
            if a_offset:
                for off in range(1, end_i + 1):
                    if off < a_offset:
                        if pd.notna(out["low"].iat[pos-off]) and pd.notna(out["sma20"].iat[pos-off]):
                            if out["low"].iat[pos-off] < out["sma20"].iat[pos-off]: pb = True
            a_above = (a_offset is not None and pd.notna(a_price) and
                       pd.notna(out["sma20"].iat[pos-a_offset]) and a_price > out["sma20"].iat[pos-a_offset])
            b_cross  = pd.notna(a_price) and out["close"].iat[pos] > a_price and out["close"].iat[pos-1] <= a_price
            b_within = pd.notna(a_price) and out["close"].iat[pos] <= a_price * (1 + max_upside_pct / 100)
            sig = (bool(out["sma200_slope_positive"].iat[pos]) and bool(out["green_candle"].iat[pos]) and
                   bool(out["sma20_above50"].iat[pos]) and a_above and bool(out["b_above20sma"].iat[pos]) and
                   pb and b_cross and b_within)
            signals.append(sig)
        out[f"{label}_signal"] = signals
    out["any_buy_signal"] = out["n1_signal"] | out["n2_signal"] | out["n3_signal"]
    return out

def add_obv_signals(df, short_sma_length=20, long_sma_length=100, confirm_days=1.0):
    out = df.copy()
    close, volume = out["close"], out["volume"]
    dv = np.where(close > close.shift(1), volume, np.where(close < close.shift(1), -volume, 0))
    out["obv"]            = pd.Series(dv, index=out.index).cumsum()
    out["obv_short_sma"]  = _sma(out["obv"], short_sma_length)
    out["obv_long_sma"]   = _sma(out["obv"], long_sma_length)
    out["long_sma_slope"] = out["obv_long_sma"] - out["obv_long_sma"].shift(1)
    out["sell_ban"] = out["long_sma_slope"] > 0
    out["buy_ban"]  = out["long_sma_slope"] < 0
    out["obv_above_both"] = (out["obv"] > out["obv_short_sma"]) & (out["obv"] > out["obv_long_sma"])
    out["obv_below_both"] = (out["obv"] < out["obv_short_sma"]) & (out["obv"] < out["obv_long_sma"])
    buy_signal = np.zeros(len(out), dtype=bool); sell_signal = np.zeros(len(out), dtype=bool)
    above_start = below_start = None; buy_triggered = sell_triggered = False
    for i in range(len(out)):
        above = bool(out["obv_above_both"].iat[i]) if pd.notna(out["obv_above_both"].iat[i]) else False
        below = bool(out["obv_below_both"].iat[i]) if pd.notna(out["obv_below_both"].iat[i]) else False
        if above:
            if above_start is None: above_start = i
            below_start = None; sell_triggered = False
        else:
            above_start = None; buy_triggered = False
        if below:
            if below_start is None: below_start = i
            above_start = None; buy_triggered = False
        else:
            below_start = None; sell_triggered = False
        def elapsed(start):
            idx = out.index
            return ((idx[i]-idx[start]).total_seconds()/86400.0 if isinstance(idx, pd.DatetimeIndex) else float(i-start))
        if above and above_start is not None and elapsed(above_start) > confirm_days:
            if not buy_triggered and not bool(out["buy_ban"].iat[i]):
                buy_signal[i] = True; buy_triggered = True
        if below and below_start is not None and elapsed(below_start) > confirm_days:
            if not sell_triggered and not bool(out["sell_ban"].iat[i]):
                sell_signal[i] = True; sell_triggered = True
    out["buy_signal"] = buy_signal; out["sell_signal"] = sell_signal
    return out


# ════════════════════════════════════════════════════════════
# LOAD DATA: memory → CSV → fresh download
# ════════════════════════════════════════════════════════════
print("=" * 55)
print("  SGX STOCK SCANNER")
print("=" * 55)

try:
    _ = sgx_stock_data, sgx_ticker_info, sti_close
    print("\n  ✅ Using in-memory SGX data.")
except NameError:
    if os.path.exists(SGX_HIST_CSV) and os.path.exists(STI_CSV):
        print("\n  📂 Loading SGX data from CSV files...")
        df_all = pd.read_csv(SGX_HIST_CSV, parse_dates=["date"])
        sgx_ticker_info = {}
        sgx_stock_data  = {}
        for sym, grp in df_all.groupby("symbol"):
            yf_t = f"{sym}.SI"
            grp  = grp.set_index("date").sort_index()
            sgx_stock_data[yf_t]  = grp[["open","high","low","close","volume"]]
            sgx_ticker_info[yf_t] = {"symbol": sym, "name": grp["name"].iloc[0], "market": grp["market"].iloc[0]}
        sti_close = pd.read_csv(STI_CSV, index_col=0, parse_dates=True).squeeze()
        print(f"  Loaded {len(sgx_stock_data)} stocks from CSV.")
    else:
        print("\n  ⬇️  CSV not found — downloading SGX data now (run SGX Download cell first to avoid this)...")
        end_date   = datetime.today().strftime("%Y-%m-%d")
        start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
        resp = requests.get("https://api.sgx.com/securities/v1.1/",
                            headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        resp.raise_for_status()
        stocks = [s for s in resp.json()["data"]["prices"] if s.get("type") == "stocks"]
        sgx_ticker_info = {f"{s['nc']}.SI": {"symbol": s["nc"], "name": s.get("n",""), "market": s.get("m","")}
                           for s in stocks if s.get("nc")}
        sti_raw   = yf.download("^STI", start=start_date, end=end_date, auto_adjust=True, progress=False)
        sti_close = sti_raw["Close"].squeeze()
        all_tickers = list(sgx_ticker_info.keys())
        batches     = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
        sgx_stock_data = {}
        for i, batch in enumerate(batches):
            print(f"  Batch {i+1}/{len(batches)}...", end=" ", flush=True)
            try:
                raw = yf.download(batch, start=start_date, end=end_date, auto_adjust=True, progress=False, threads=True)
                if len(batch) == 1:
                    if not raw.empty: sgx_stock_data[batch[0]] = raw.rename(columns=str.lower)
                else:
                    for t in batch:
                        try:
                            df = raw.xs(t, level=1, axis=1).dropna(how="all")
                            if not df.empty: sgx_stock_data[t] = df.rename(columns=str.lower)
                        except KeyError: pass
                print(f"ok ({sum(1 for t in batch if t in sgx_stock_data)}/{len(batch)})")
            except Exception as e:
                print(f"ERROR: {e}")
            time.sleep(0.3)


# ════════════════════════════════════════════════════════════
# FILTER + SCAN
# ════════════════════════════════════════════════════════════
filtered = {t: df for t, df in sgx_stock_data.items()
            if float(df["volume"].iloc[-1]) > MIN_VOLUME and float(df["close"].iloc[-1]) > MIN_PRICE}
print(f"\n  Filter (vol>{MIN_VOLUME:,} & price>S${MIN_PRICE}): {len(sgx_stock_data)} → {len(filtered)} stocks")

results = []
for idx, (yf_t, df) in enumerate(filtered.items()):
    if len(df) < 110: continue
    info = sgx_ticker_info.get(yf_t, {})
    try:
        sma_sig      = bool(add_sma_gap_rs_buy_sell_signals(df, sti_close)["buy_signal"].iloc[-1])
        bo_sig       = bool(add_breakout_buy_signals(df)["any_buy_signal"].iloc[-1])
        obv_sell_ban = bool(add_obv_signals(df)["sell_ban"].iloc[-1])
        confirmed    = (sma_sig and obv_sell_ban) or (bo_sig and obv_sell_ban)
        results.append({"symbol": info.get("symbol",""), "name": info.get("name",""), "market": info.get("market",""),
                        "signal_confirmed": confirmed, "sma_gap_buy": sma_sig, "breakout_buy": bo_sig,
                        "obv_sell_ban": obv_sell_ban, "last_close": round(float(df["close"].iloc[-1]),4),
                        "last_date": df.index[-1].strftime("%Y-%m-%d"), "bars": len(df)})
    except Exception: pass
    if (idx+1) % 50 == 0:
        print(f"  {idx+1}/{len(filtered)} scanned | {sum(1 for r in results if r['signal_confirmed'])} confirmed so far")

matched = [r for r in results if r["signal_confirmed"]]
print(f"\n  Scan complete: {len(results)} scanned, {len(matched)} confirmed signals")

# Export
fieldnames = ["symbol","name","market","signal_confirmed","sma_gap_buy","breakout_buy","obv_sell_ban","last_close","last_date","bars"]
with open(OUTPUT_RESULTS, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames); w.writeheader(); w.writerows(sorted(results, key=lambda r: r["signal_confirmed"], reverse=True))
with open(OUTPUT_WATCHLIST, "w", encoding="utf-8") as f:
    [f.write(f"SGX:{r['symbol']}\n") for r in matched]

print("\n" + "=" * 60)
print(f"  CONFIRMED SIGNALS ({len(matched)} stocks)")
print("=" * 60)
if matched:
    print(f"  {'Symbol':<8} {'Name':<30} {'Market':<12} {'SMA':^5} {'BO':^5} {'OBV-SB':^7}")
    for r in matched:
        print(f"  {r['symbol']:<8} {r['name'][:30]:<30} {r['market']:<12} "
              f"{'Y' if r['sma_gap_buy'] else 'N':^5} {'Y' if r['breakout_buy'] else 'N':^5} {'Y' if r['obv_sell_ban'] else 'N':^7}")
else:
    print("  No confirmed signals today.")
print(f"\n  Results → {OUTPUT_RESULTS}")
print(f"  TV list → {OUTPUT_WATCHLIST}")

## Cell 3b — SGX 20 SMA Crossup Scanner
Scans SGX stocks for **all three** conditions on the latest bar:
1. Price **just crossed up** the 20-day SMA
2. **Green candle** (close > open)
3. **RS-Delta histogram green** — `(close/sma30) − (STI/sma30) ≥ 0`

**Loads from memory → CSV → downloads fresh** (no re-download if Cell 2 was run).

In [ ]:
# ============================================================
# SGX 20 SMA CROSSUP SCANNER
# Conditions (all must be true on latest bar):
#   1. Price just crossed UP the 20-day SMA
#   2. Green candle (close > open)
#   3. RS-Delta histogram green: (close/sma30) − (STI/sma30) >= 0
# Filter: volume > 100,000 AND price > S$0.20
# Loads: memory → sgx_historical.csv → fresh download
# ============================================================
import csv as _csv, os, time, warnings
import numpy as np, pandas as pd, requests, yfinance as yf
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")

SGX_HIST_CSV = "/content/sgx_historical.csv"
STI_CSV      = "/content/sti_close.csv"
OUTPUT_CSV   = "/content/sgx_sma20_crossup.csv"
OUTPUT_WL    = "/content/sgx_sma20_crossup_watchlist.txt"
BATCH_SIZE   = 50
MIN_VOLUME, MIN_PRICE = 100_000, 0.20
SMA20_LEN, RS_SMA_LEN = 20, 30

print("=" * 58)
print("  SGX 20 SMA CROSSUP SCANNER")
print("=" * 58)

# ── Load data ────────────────────────────────────────────────
try:
    _ = sgx_stock_data, sgx_ticker_info, sti_close
    print("\n  ✅ Using in-memory SGX data.")
except NameError:
    if os.path.exists(SGX_HIST_CSV) and os.path.exists(STI_CSV):
        print("\n  📂 Loading from CSV...")
        df_all = pd.read_csv(SGX_HIST_CSV, parse_dates=["date"])
        sgx_ticker_info = {}; sgx_stock_data = {}
        for sym, grp in df_all.groupby("symbol"):
            yf_t = f"{sym}.SI"; grp = grp.set_index("date").sort_index()
            sgx_stock_data[yf_t]  = grp[["open","high","low","close","volume"]]
            sgx_ticker_info[yf_t] = {"symbol": sym, "name": grp["name"].iloc[0], "market": grp["market"].iloc[0]}
        sti_close = pd.read_csv(STI_CSV, index_col=0, parse_dates=True).squeeze()
        print(f"  Loaded {len(sgx_stock_data)} stocks.")
    else:
        print("\n  ⬇️  Downloading SGX data (run Cell 2 first to avoid this)...")
        end_date   = datetime.today().strftime("%Y-%m-%d")
        start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
        resp = requests.get("https://api.sgx.com/securities/v1.1/", headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        resp.raise_for_status()
        stocks = [s for s in resp.json()["data"]["prices"] if s.get("type") == "stocks"]
        sgx_ticker_info = {f"{s['nc']}.SI": {"symbol": s["nc"], "name": s.get("n",""), "market": s.get("m","")} for s in stocks if s.get("nc")}
        sti_raw   = yf.download("^STI", start=start_date, end=end_date, auto_adjust=True, progress=False)
        sti_close = sti_raw["Close"].squeeze()
        all_tickers = list(sgx_ticker_info.keys())
        batches = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
        sgx_stock_data = {}
        for i, batch in enumerate(batches):
            print(f"  Batch {i+1}/{len(batches)}...", end=" ", flush=True)
            try:
                raw = yf.download(batch, start=start_date, end=end_date, auto_adjust=True, progress=False, threads=True)
                if len(batch) == 1:
                    if not raw.empty: sgx_stock_data[batch[0]] = raw.rename(columns=str.lower)
                else:
                    for t in batch:
                        try:
                            df = raw.xs(t, level=1, axis=1).dropna(how="all")
                            if not df.empty: sgx_stock_data[t] = df.rename(columns=str.lower)
                        except KeyError: pass
                print(f"ok ({sum(1 for t in batch if t in sgx_stock_data)}/{len(batch)})")
            except Exception as e: print(f"ERROR: {e}")
            time.sleep(0.3)

# ── STI baseline ─────────────────────────────────────────────
sti_sma      = sti_close.rolling(RS_SMA_LEN, min_periods=RS_SMA_LEN).mean()
sti_result_b = sti_close / sti_sma

# ── Filter + Scan ────────────────────────────────────────────
filtered = {t: df for t, df in sgx_stock_data.items()
            if float(df["volume"].iloc[-1]) > MIN_VOLUME and float(df["close"].iloc[-1]) > MIN_PRICE}
print(f"\n  Filter (vol>{MIN_VOLUME:,} & price>S${MIN_PRICE}): {len(sgx_stock_data)} → {len(filtered)} stocks")

results = []; confirmed = []
for yf_t, df in filtered.items():
    if len(df) < RS_SMA_LEN + 2: continue
    info  = sgx_ticker_info.get(yf_t, {})
    close = df["close"]; open_ = df["open"]
    sma20 = close.rolling(SMA20_LEN, min_periods=SMA20_LEN).mean()
    if pd.isna(sma20.iloc[-1]) or pd.isna(sma20.iloc[-2]): continue
    cross_up     = (close.iloc[-2] <= sma20.iloc[-2]) and (close.iloc[-1] > sma20.iloc[-1])
    green_candle = float(close.iloc[-1]) > float(open_.iloc[-1])
    result_a  = close / close.rolling(RS_SMA_LEN, min_periods=RS_SMA_LEN).mean()
    delta_ab  = result_a - sti_result_b.reindex(close.index)
    if pd.isna(delta_ab.iloc[-1]): continue
    rs_green  = float(delta_ab.iloc[-1]) >= 0
    signal    = cross_up and green_candle and rs_green
    row = {"symbol": info.get("symbol",""), "name": info.get("name",""), "market": info.get("market",""),
           "signal": signal, "last_close": round(float(close.iloc[-1]),4),
           "sma20": round(float(sma20.iloc[-1]),4),
           "pct_vs_sma20": round((float(close.iloc[-1])/float(sma20.iloc[-1])-1)*100,2),
           "delta_ab": round(float(delta_ab.iloc[-1]),6),
           "cross_up": cross_up, "green_candle": green_candle, "rs_green": rs_green,
           "last_date": df.index[-1].strftime("%Y-%m-%d")}
    results.append(row)
    if signal: confirmed.append(row)

print(f"  Scanned {len(results)} stocks | {len(confirmed)} confirmed\n")
print("=" * 80)
print(f"  SGX 20 SMA CROSSUP — CONFIRMED ({len(confirmed)} stocks)")
print("=" * 80)
if confirmed:
    print(f"  {'Symbol':<8} {'Name':<30} {'Market':<12} {'Close':>8} {'%vsSMA20':>9} {'DeltaAB':>10}")
    print(f"  {'-'*8} {'-'*30} {'-'*12} {'-'*8} {'-'*9} {'-'*10}")
    for r in sorted(confirmed, key=lambda x: -x["delta_ab"]):
        print(f"  {r['symbol']:<8} {r['name'][:30]:<30} {r['market']:<12} "
              f"{r['last_close']:>8.4f} {r['pct_vs_sma20']:>+8.2f}% {r['delta_ab']:>10.6f}")
else:
    print("  No stocks matched today.")
print("=" * 80)

fieldnames = ["symbol","name","market","signal","last_close","sma20","pct_vs_sma20","delta_ab","cross_up","green_candle","rs_green","last_date"]
with open(OUTPUT_CSV,"w",newline="",encoding="utf-8") as f:
    w = _csv.DictWriter(f, fieldnames=fieldnames); w.writeheader()
    w.writerows(sorted(results, key=lambda r: r["signal"], reverse=True))
with open(OUTPUT_WL,"w",encoding="utf-8") as f:
    [f.write(f"SGX:{r['symbol']}\n") for r in confirmed]
print(f"\n  Results → {OUTPUT_CSV}   |   TV list → {OUTPUT_WL}")

---
## Cell 4 — S&P 500 Download
Downloads all S&P 500 stocks + SPY benchmark. Saves to `/content/`. Run once, then use the scanners without re-downloading.

In [ ]:
# ============================================================
# S&P 500 DOWNLOAD — saves sp500_historical.csv + spy_close.csv
# Run once per session. Scanner cells load from these files.
# ============================================================
import csv, io, time, warnings
import requests, pandas as pd, yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

SP500_HIST_CSV = "/content/sp500_historical.csv"
SPY_CSV        = "/content/spy_close.csv"
BATCH_SIZE     = 50

print("=" * 55)
print("  S&P 500 DATA DOWNLOAD")
print("=" * 55)

# ── Fetch ticker list from Wikipedia ────────────────────────
print("\n[1/3] Fetching S&P 500 tickers from Wikipedia...")
html  = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                     headers={"User-Agent": "Mozilla/5.0"}).text
sp500 = pd.read_html(io.StringIO(html), flavor="lxml")[0]
sp500["yf_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500_ticker_info  = {
    row["yf_ticker"]: {"symbol": row["Symbol"], "name": row["Security"], "sector": row["GICS Sector"]}
    for _, row in sp500.iterrows()
}
print(f"    Found {len(sp500_ticker_info)} stocks")

# ── Download price history ───────────────────────────────────
end_date   = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
print(f"\n[2/3] Downloading data ({start_date} → {end_date})...")

print("    Fetching SPY benchmark...")
spy_raw   = yf.download("SPY", start=start_date, end=end_date, auto_adjust=True, progress=False)
spy_close = spy_raw["Close"].squeeze()
spy_close.to_csv(SPY_CSV)

all_tickers    = list(sp500_ticker_info.keys())
batches        = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
sp500_stock_data = {}

with open(SP500_HIST_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["symbol", "name", "sector", "date", "open", "high", "low", "close", "volume"])
    for i, batch in enumerate(batches):
        print(f"    Batch {i+1}/{len(batches)}...", end=" ", flush=True)
        try:
            raw = yf.download(batch, start=start_date, end=end_date,
                              auto_adjust=True, progress=False, threads=True)
            results = {}
            if len(batch) == 1:
                if not raw.empty: results[batch[0]] = raw.rename(columns=str.lower)
            else:
                for t in batch:
                    try:
                        df = raw.xs(t, level=1, axis=1).dropna(how="all")
                        if not df.empty: results[t] = df.rename(columns=str.lower)
                    except KeyError: pass
            for yf_t, df in results.items():
                info = sp500_ticker_info[yf_t]
                sp500_stock_data[yf_t] = df
                for date, row in df.iterrows():
                    writer.writerow([
                        info["symbol"], info["name"], info["sector"],
                        date.strftime("%Y-%m-%d"),
                        round(float(row.get("open",  0) or 0), 4),
                        round(float(row.get("high",  0) or 0), 4),
                        round(float(row.get("low",   0) or 0), 4),
                        round(float(row.get("close", 0) or 0), 4),
                        int(row.get("volume", 0) or 0),
                    ])
            print(f"ok ({len(results)}/{len(batch)})")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.3)

print(f"\n[3/3] Done!")
print(f"    Stocks downloaded : {len(sp500_stock_data)}")
print(f"    Saved to          : {SP500_HIST_CSV}")
print(f"    SPY benchmark     : {SPY_CSV}")
print("    ✅ Scanner cells can now use this data without re-downloading.")

## Cell 5 — S&P 500 Stock Scanner
Scans S&P 500 stocks for SMA Gap, Breakout, and OBV signals.  
**Loads from memory → CSV → downloads fresh** (no re-download if Cell 4 was run).

In [ ]:
# ============================================================
# S&P 500 STOCK SCANNER
# Loads data from: memory → sp500_historical.csv → fresh download
# ============================================================
import csv, io, os, time, warnings
import numpy as np, pandas as pd, requests, yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

SP500_HIST_CSV   = "/content/sp500_historical.csv"
SPY_CSV          = "/content/spy_close.csv"
OUTPUT_RESULTS   = "/content/sp500_scan_results.csv"
OUTPUT_WATCHLIST = "/content/sp500_watchlist.txt"
BATCH_SIZE       = 50


# ════════════════════════════════════════════════════════════
# SCANNER FUNCTIONS
# ════════════════════════════════════════════════════════════
def _sma(series, length):
    return series.rolling(length, min_periods=length).mean()

def add_sma_gap_rs_buy_sell_signals(df, index_close,
        b_sma20_len=20, b_sma50_len=50, b_slope_lookback=5,
        b_slope_smooth_len=1, a_rs_sma_length=30, a_delta_ma_length=9):
    out = df.copy()
    index_close = index_close.reindex(out.index).astype(float)
    out["b_sma20"]       = _sma(out["close"], b_sma20_len)
    out["b_sma50"]       = _sma(out["close"], b_sma50_len)
    out["b_raw_slope20"] = (out["b_sma20"] - out["b_sma20"].shift(b_slope_lookback)) / out["b_sma20"].shift(b_slope_lookback) * 100.0
    out["b_raw_slope50"] = (out["b_sma50"] - out["b_sma50"].shift(b_slope_lookback)) / out["b_sma50"].shift(b_slope_lookback) * 100.0
    out["b_slope20"]     = _sma(out["b_raw_slope20"], b_slope_smooth_len)
    out["b_slope50"]     = _sma(out["b_raw_slope50"], b_slope_smooth_len)
    out["b_bull"]        = out["b_slope20"] > out["b_slope50"]
    stock_sma = _sma(out["close"], a_rs_sma_length); index_sma = _sma(index_close, a_rs_sma_length)
    out["delta_ab"]  = (out["close"] / stock_sma) - (index_close / index_sma)
    out["delta_ma"]  = _sma(out["delta_ab"], a_delta_ma_length)
    out["slope_gap"] = out["b_slope20"] - out["b_slope50"]
    out["a_two_day_green_rising"] = ((out["delta_ab"] >= 0) & (out["delta_ab"].shift(1) >= 0) & (out["delta_ab"] > out["delta_ab"].shift(1)))
    out["green_candle"]       = out["close"] > out["open"]
    out["red_candle"]         = out["close"] < out["open"]
    out["slope_gap_reducing"] = (out["b_slope20"] > out["b_slope50"]) & (out["slope_gap"] < out["slope_gap"].shift(1))
    out["buy_condition"]  = out["b_bull"] & out["a_two_day_green_rising"] & out["green_candle"]
    out["sell_condition"] = (out["delta_ab"] < out["delta_ma"]) & out["slope_gap_reducing"] & out["red_candle"]
    buy_signal = np.zeros(len(out), dtype=bool); sell_signal = np.zeros(len(out), dtype=bool)
    in_position = buy_ban = sell_ban = False
    for i in range(len(out)):
        buy  = bool(out["buy_condition"].iat[i])  and not buy_ban  and not in_position
        sell = bool(out["sell_condition"].iat[i]) and not sell_ban and in_position
        if buy:  buy_signal[i]  = True; in_position = True;  buy_ban = True;  sell_ban = False
        if sell: sell_signal[i] = True; in_position = False; sell_ban = True; buy_ban  = False
    out["buy_signal"] = buy_signal; out["sell_signal"] = sell_signal
    return out

def add_breakout_buy_signals(df, lookback_n1=30, lookback_n2=60, lookback_n3=90,
        sma200_slope_lookback=5, max_upside_pct=5.0):
    out = df.copy()
    out["sma20"]  = _sma(out["close"], 20); out["sma50"] = _sma(out["close"], 50); out["sma200"] = _sma(out["close"], 200)
    out["sma200_slope_positive"] = out["sma200"] > out["sma200"].shift(sma200_slope_lookback)
    out["green_candle"] = out["close"] > out["open"]; out["sma20_above50"] = out["sma20"] > out["sma50"]; out["b_above20sma"] = out["close"] > out["sma20"]
    for label, lookback in [("n1", lookback_n1), ("n2", lookback_n2), ("n3", lookback_n3)]:
        signals = []
        for pos in range(len(out)):
            if pos < 1: signals.append(False); continue
            end_i = min(lookback, pos); a_price, a_offset = float("nan"), None
            for off in range(1, end_i+1):
                h = out["high"].iat[pos-off]
                if pd.notna(h) and (pd.isna(a_price) or h > a_price): a_price, a_offset = h, off
            pb = False
            if a_offset:
                for off in range(1, end_i+1):
                    if off < a_offset and pd.notna(out["low"].iat[pos-off]) and pd.notna(out["sma20"].iat[pos-off]):
                        if out["low"].iat[pos-off] < out["sma20"].iat[pos-off]: pb = True
            a_above = (a_offset is not None and pd.notna(a_price) and pd.notna(out["sma20"].iat[pos-a_offset]) and a_price > out["sma20"].iat[pos-a_offset])
            b_cross  = pd.notna(a_price) and out["close"].iat[pos] > a_price and out["close"].iat[pos-1] <= a_price
            b_within = pd.notna(a_price) and out["close"].iat[pos] <= a_price * (1 + max_upside_pct/100)
            signals.append(bool(out["sma200_slope_positive"].iat[pos]) and bool(out["green_candle"].iat[pos]) and
                           bool(out["sma20_above50"].iat[pos]) and a_above and bool(out["b_above20sma"].iat[pos]) and pb and b_cross and b_within)
        out[f"{label}_signal"] = signals
    out["any_buy_signal"] = out["n1_signal"] | out["n2_signal"] | out["n3_signal"]
    return out

def add_obv_signals(df, short_sma_length=20, long_sma_length=100, confirm_days=1.0):
    out = df.copy(); close, volume = out["close"], out["volume"]
    dv = np.where(close > close.shift(1), volume, np.where(close < close.shift(1), -volume, 0))
    out["obv"] = pd.Series(dv, index=out.index).cumsum()
    out["obv_short_sma"] = _sma(out["obv"], short_sma_length); out["obv_long_sma"] = _sma(out["obv"], long_sma_length)
    out["long_sma_slope"] = out["obv_long_sma"] - out["obv_long_sma"].shift(1)
    out["sell_ban"] = out["long_sma_slope"] > 0; out["buy_ban"] = out["long_sma_slope"] < 0
    out["obv_above_both"] = (out["obv"] > out["obv_short_sma"]) & (out["obv"] > out["obv_long_sma"])
    out["obv_below_both"] = (out["obv"] < out["obv_short_sma"]) & (out["obv"] < out["obv_long_sma"])
    buy_signal = np.zeros(len(out), dtype=bool); sell_signal = np.zeros(len(out), dtype=bool)
    above_start = below_start = None; buy_triggered = sell_triggered = False
    for i in range(len(out)):
        above = bool(out["obv_above_both"].iat[i]) if pd.notna(out["obv_above_both"].iat[i]) else False
        below = bool(out["obv_below_both"].iat[i]) if pd.notna(out["obv_below_both"].iat[i]) else False
        if above:
            if above_start is None: above_start = i
            below_start = None; sell_triggered = False
        else: above_start = None; buy_triggered = False
        if below:
            if below_start is None: below_start = i
            above_start = None; buy_triggered = False
        else: below_start = None; sell_triggered = False
        def elapsed(start):
            idx = out.index
            return ((idx[i]-idx[start]).total_seconds()/86400.0 if isinstance(idx, pd.DatetimeIndex) else float(i-start))
        if above and above_start is not None and elapsed(above_start) > confirm_days:
            if not buy_triggered and not bool(out["buy_ban"].iat[i]): buy_signal[i] = True; buy_triggered = True
        if below and below_start is not None and elapsed(below_start) > confirm_days:
            if not sell_triggered and not bool(out["sell_ban"].iat[i]): sell_signal[i] = True; sell_triggered = True
    out["buy_signal"] = buy_signal; out["sell_signal"] = sell_signal
    return out


# ════════════════════════════════════════════════════════════
# LOAD DATA: memory → CSV → fresh download
# ════════════════════════════════════════════════════════════
print("=" * 55)
print("  S&P 500 STOCK SCANNER")
print("=" * 55)

try:
    _ = sp500_stock_data, sp500_ticker_info, spy_close
    print("\n  ✅ Using in-memory S&P 500 data.")
except NameError:
    if os.path.exists(SP500_HIST_CSV) and os.path.exists(SPY_CSV):
        print("\n  📂 Loading S&P 500 data from CSV files...")
        df_all = pd.read_csv(SP500_HIST_CSV, parse_dates=["date"])
        sp500_ticker_info = {}
        sp500_stock_data  = {}
        for sym, grp in df_all.groupby("symbol"):
            yf_t = sym.replace(".", "-")
            grp  = grp.set_index("date").sort_index()
            sp500_stock_data[yf_t]  = grp[["open","high","low","close","volume"]]
            sp500_ticker_info[yf_t] = {"symbol": sym, "name": grp["name"].iloc[0], "sector": grp["sector"].iloc[0]}
        spy_close = pd.read_csv(SPY_CSV, index_col=0, parse_dates=True).squeeze()
        print(f"  Loaded {len(sp500_stock_data)} stocks from CSV.")
    else:
        print("\n  ⬇️  CSV not found — downloading S&P 500 data now (run S&P 500 Download cell first to avoid this)...")
        end_date   = datetime.today().strftime("%Y-%m-%d")
        start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
        html  = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={"User-Agent": "Mozilla/5.0"}).text
        sp500 = pd.read_html(io.StringIO(html), flavor="lxml")[0]
        sp500["yf_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
        sp500_ticker_info  = {row["yf_ticker"]: {"symbol": row["Symbol"], "name": row["Security"], "sector": row["GICS Sector"]} for _, row in sp500.iterrows()}
        spy_raw   = yf.download("SPY", start=start_date, end=end_date, auto_adjust=True, progress=False)
        spy_close = spy_raw["Close"].squeeze()
        all_tickers = list(sp500_ticker_info.keys())
        batches     = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
        sp500_stock_data = {}
        for i, batch in enumerate(batches):
            print(f"  Batch {i+1}/{len(batches)}...", end=" ", flush=True)
            try:
                raw = yf.download(batch, start=start_date, end=end_date, auto_adjust=True, progress=False, threads=True)
                if len(batch) == 1:
                    if not raw.empty: sp500_stock_data[batch[0]] = raw.rename(columns=str.lower)
                else:
                    for t in batch:
                        try:
                            df = raw.xs(t, level=1, axis=1).dropna(how="all")
                            if not df.empty: sp500_stock_data[t] = df.rename(columns=str.lower)
                        except KeyError: pass
                print(f"ok ({sum(1 for t in batch if t in sp500_stock_data)}/{len(batch)})")
            except Exception as e: print(f"ERROR: {e}")
            time.sleep(0.3)


# ════════════════════════════════════════════════════════════
# SCAN
# ════════════════════════════════════════════════════════════
print(f"\n  Scanning {len(sp500_stock_data)} stocks...")
results = []
for idx, (yf_t, df) in enumerate(sp500_stock_data.items()):
    if len(df) < 110: continue
    info = sp500_ticker_info.get(yf_t, {})
    try:
        sma_sig      = bool(add_sma_gap_rs_buy_sell_signals(df, spy_close)["buy_signal"].iloc[-1])
        bo_sig       = bool(add_breakout_buy_signals(df)["any_buy_signal"].iloc[-1])
        obv_sell_ban = bool(add_obv_signals(df)["sell_ban"].iloc[-1])
        confirmed    = (sma_sig and obv_sell_ban) or (bo_sig and obv_sell_ban)
        results.append({"symbol": info.get("symbol",""), "name": info.get("name",""), "sector": info.get("sector",""),
                        "signal_confirmed": confirmed, "sma_gap_buy": sma_sig, "breakout_buy": bo_sig,
                        "obv_sell_ban": obv_sell_ban, "last_close": round(float(df["close"].iloc[-1]),4),
                        "last_date": df.index[-1].strftime("%Y-%m-%d"), "bars": len(df)})
    except Exception: pass
    if (idx+1) % 50 == 0: print(f"  {idx+1}/{len(sp500_stock_data)} scanned | {sum(1 for r in results if r['signal_confirmed'])} confirmed so far")

matched = [r for r in results if r["signal_confirmed"]]
print(f"\n  Scan complete: {len(results)} scanned, {len(matched)} confirmed signals")

fieldnames = ["symbol","name","sector","signal_confirmed","sma_gap_buy","breakout_buy","obv_sell_ban","last_close","last_date","bars"]
with open(OUTPUT_RESULTS,"w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames); w.writeheader(); w.writerows(sorted(results, key=lambda r: r["signal_confirmed"], reverse=True))
with open(OUTPUT_WATCHLIST,"w",encoding="utf-8") as f:
    [f.write(f"{r['symbol']}\n") for r in matched]

print("\n" + "=" * 65)
print(f"  CONFIRMED SIGNALS ({len(matched)} stocks)")
print("=" * 65)
if matched:
    print(f"  {'Symbol':<8} {'Name':<28} {'Sector':<22} {'SMA':^5} {'BO':^5} {'OBV-SB':^7}")
    for r in sorted(matched, key=lambda x: x["sector"]):
        print(f"  {r['symbol']:<8} {r['name'][:28]:<28} {r['sector'][:22]:<22} {'Y' if r['sma_gap_buy'] else 'N':^5} {'Y' if r['breakout_buy'] else 'N':^5} {'Y' if r['obv_sell_ban'] else 'N':^7}")
else:
    print("  No confirmed signals today.")
print(f"\n  Results → {OUTPUT_RESULTS}   |   TV list → {OUTPUT_WATCHLIST}")

## Cell 5b — S&P 500 20 SMA Crossup Scanner
Scans S&P 500 stocks for **all three** conditions on the latest bar:
1. Price **just crossed up** the 20-day SMA
2. **Green candle** (close > open)
3. **RS-Delta histogram green** — `(close/sma30) − (SPY/sma30) ≥ 0`

**Loads from memory → CSV → downloads fresh** (no re-download if Cell 4 was run).

In [10]:
# ============================================================
# S&P 500 20 SMA CROSSUP SCANNER
# Conditions (all must be true on latest bar):
#   1. Price just crossed UP the 20-day SMA
#   2. Green candle (close > open)
#   3. RS-Delta histogram green: (close/sma30) − (SPY/sma30) >= 0
# Loads: memory → sp500_historical.csv → fresh download
# ============================================================
import csv as _csv, io, os, time, warnings
import numpy as np, pandas as pd, requests, yfinance as yf
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")

SP500_HIST_CSV = "/content/sp500_historical.csv"
SPY_CSV        = "/content/spy_close.csv"
OUTPUT_CSV     = "/content/sp500_sma20_crossup.csv"
OUTPUT_WL      = "/content/sp500_sma20_crossup_watchlist.txt"
BATCH_SIZE     = 50
SMA20_LEN, RS_SMA_LEN = 20, 30

print("=" * 60)
print("  S&P 500 20 SMA CROSSUP SCANNER")
print("=" * 60)

# ── Load data ────────────────────────────────────────────────
try:
    _ = sp500_stock_data, sp500_ticker_info, spy_close
    print("\n  ✅ Using in-memory S&P 500 data.")
except NameError:
    if os.path.exists(SP500_HIST_CSV) and os.path.exists(SPY_CSV):
        print("\n  📂 Loading from CSV...")
        df_all = pd.read_csv(SP500_HIST_CSV, parse_dates=["date"])
        sp500_ticker_info = {}; sp500_stock_data = {}
        for sym, grp in df_all.groupby("symbol"):
            yf_t = sym.replace(".", "-"); grp = grp.set_index("date").sort_index()
            sp500_stock_data[yf_t]  = grp[["open","high","low","close","volume"]]
            sp500_ticker_info[yf_t] = {"symbol": sym, "name": grp["name"].iloc[0], "sector": grp["sector"].iloc[0]}
        spy_close = pd.read_csv(SPY_CSV, index_col=0, parse_dates=True).squeeze()
        print(f"  Loaded {len(sp500_stock_data)} stocks.")
    else:
        print("\n  ⬇️  Downloading S&P 500 data (run Cell 4 first to avoid this)...")
        end_date   = datetime.today().strftime("%Y-%m-%d")
        start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
        html  = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={"User-Agent": "Mozilla/5.0"}).text
        sp500 = pd.read_html(io.StringIO(html), flavor="lxml")[0]
        sp500["yf_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
        sp500_ticker_info  = {row["yf_ticker"]: {"symbol": row["Symbol"], "name": row["Security"], "sector": row["GICS Sector"]} for _, row in sp500.iterrows()}
        spy_raw   = yf.download("SPY", start=start_date, end=end_date, auto_adjust=True, progress=False)
        spy_close = spy_raw["Close"].squeeze()
        all_tickers = list(sp500_ticker_info.keys())
        batches = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
        sp500_stock_data = {}
        for i, batch in enumerate(batches):
            print(f"  Batch {i+1}/{len(batches)}...", end=" ", flush=True)
            try:
                raw = yf.download(batch, start=start_date, end=end_date, auto_adjust=True, progress=False, threads=True)
                if len(batch) == 1:
                    if not raw.empty: sp500_stock_data[batch[0]] = raw.rename(columns=str.lower)
                else:
                    for t in batch:
                        try:
                            df = raw.xs(t, level=1, axis=1).dropna(how="all")
                            if not df.empty: sp500_stock_data[t] = df.rename(columns=str.lower)
                        except KeyError: pass
                print(f"ok ({sum(1 for t in batch if t in sp500_stock_data)}/{len(batch)})")
            except Exception as e: print(f"ERROR: {e}")
            time.sleep(0.3)

# ── SPY baseline ─────────────────────────────────────────────
spy_sma      = spy_close.rolling(RS_SMA_LEN, min_periods=RS_SMA_LEN).mean()
spy_result_b = spy_close / spy_sma

# ── Scan ─────────────────────────────────────────────────────
print(f"\n  Scanning {len(sp500_stock_data)} stocks...")
results = []; confirmed = []
for yf_t, df in sp500_stock_data.items():
    if len(df) < RS_SMA_LEN + 2: continue
    info  = sp500_ticker_info.get(yf_t, {})
    close = df["close"]; open_ = df["open"]
    sma20 = close.rolling(SMA20_LEN, min_periods=SMA20_LEN).mean()
    if pd.isna(sma20.iloc[-1]) or pd.isna(sma20.iloc[-2]): continue
    cross_up     = (close.iloc[-2] <= sma20.iloc[-2]) and (close.iloc[-1] > sma20.iloc[-1])
    green_candle = float(close.iloc[-1]) > float(open_.iloc[-1])
    result_a  = close / close.rolling(RS_SMA_LEN, min_periods=RS_SMA_LEN).mean()
    delta_ab  = result_a - spy_result_b.reindex(close.index)
    if pd.isna(delta_ab.iloc[-1]): continue
    rs_green  = float(delta_ab.iloc[-1]) >= 0
    signal    = cross_up and green_candle and rs_green
    row = {"symbol": info.get("symbol",""), "name": info.get("name",""), "sector": info.get("sector",""),
           "signal": signal, "last_close": round(float(close.iloc[-1]),4),
           "sma20": round(float(sma20.iloc[-1]),4),
           "pct_vs_sma20": round((float(close.iloc[-1])/float(sma20.iloc[-1])-1)*100,2),
           "delta_ab": round(float(delta_ab.iloc[-1]),6),
           "cross_up": cross_up, "green_candle": green_candle, "rs_green": rs_green,
           "last_date": df.index[-1].strftime("%Y-%m-%d")}
    results.append(row)
    if signal: confirmed.append(row)

print(f"  Scanned {len(results)} stocks | {len(confirmed)} confirmed\n")
print("=" * 90)
print(f"  S&P 500 20 SMA CROSSUP — CONFIRMED ({len(confirmed)} stocks)")
print("=" * 90)
if confirmed:
    print(f"  {'Symbol':<8} {'Name':<28} {'Sector':<24} {'Close':>8} {'%vsSMA20':>9} {'DeltaAB':>10}")
    print(f"  {'-'*8} {'-'*28} {'-'*24} {'-'*8} {'-'*9} {'-'*10}")
    for r in sorted(confirmed, key=lambda x: -x["delta_ab"]):
        print(f"  {r['symbol']:<8} {r['name'][:28]:<28} {r['sector'][:24]:<24} "
              f"{r['last_close']:>8.4f} {r['pct_vs_sma20']:>+8.2f}% {r['delta_ab']:>10.6f}")
else:
    print("  No stocks matched today.")
print("=" * 90)
print(f"\n  Legend: %vsSMA20 = how far close is above 20 SMA | DeltaAB = RS vs SPY (higher = stronger)")

fieldnames = ["symbol","name","sector","signal","last_close","sma20","pct_vs_sma20","delta_ab","cross_up","green_candle","rs_green","last_date"]
with open(OUTPUT_CSV,"w",newline="",encoding="utf-8") as f:
    w = _csv.DictWriter(f, fieldnames=fieldnames); w.writeheader()
    w.writerows(sorted(results, key=lambda r: r["signal"], reverse=True))
with open(OUTPUT_WL,"w",encoding="utf-8") as f:
    [f.write(f"{r['symbol']}\n") for r in confirmed]
print(f"\n  Results → {OUTPUT_CSV}   |   TV list → {OUTPUT_WL}")

  S&P 500 20 SMA CROSSUP SCANNER

  ✅ Using in-memory S&P 500 data.

  Scanning 503 stocks...
  Scanned 503 stocks | 1 confirmed

  S&P 500 20 SMA CROSSUP — CONFIRMED (1 stocks)
  Symbol   Name                         Sector                      Close  %vsSMA20    DeltaAB
  -------- ---------------------------- ------------------------ -------- --------- ----------
  F        Ford Motor Company           Consumer Discretionary    13.5700   +11.11%   0.067445

  Legend: %vsSMA20 = how far close is above 20 SMA | DeltaAB = RS vs SPY (higher = stronger)

  Results → /content/sp500_sma20_crossup.csv   |   TV list → /content/sp500_sma20_crossup_watchlist.txt


## Cell 6 — S&P 500 RSI Strength + 52-Week High Scanner
Finds stocks where RSI(14) beats SPY by ≥5 pts, price within 2% of 52-week high, and within ±10% of 50-day SMA.  
**Loads from memory → CSV → downloads fresh** (no re-download if Cell 4 was run).

In [ ]:
# ============================================================
# S&P 500 RSI STRENGTH + 52-WEEK HIGH SCANNER
# Loads data from: memory → sp500_historical.csv → fresh download
#
# Filters (ALL three must pass):
#   1. RSI(14) >= SPY RSI(14) + 5 pts
#   2. Price within 2% of 52-week high
#   3. Price within ±10% of 50-day SMA
# ============================================================
import csv as _csv, io, os, time, warnings
import numpy as np, pandas as pd, requests, yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

SP500_HIST_CSV   = "/content/sp500_historical.csv"
SPY_CSV          = "/content/spy_close.csv"
OUTPUT_RSI_CSV   = "/content/sp500_rsi_scan.csv"
OUTPUT_RSI_WL    = "/content/sp500_rsi_watchlist.txt"
BATCH_SIZE       = 50

RSI_PERIOD      = 14
RSI_MIN_EDGE    = 5.0
HIGH_52W_BUFFER = 0.02
SMA50_BAND      = 0.10

def _rsi(series, length=14):
    delta = series.diff(); gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=length-1, min_periods=length).mean()
    avg_loss = loss.ewm(com=length-1, min_periods=length).mean()
    rs = avg_gain / avg_loss.replace(0, float("nan"))
    return 100 - (100 / (1 + rs))

print("=" * 70)
print("  S&P 500 RSI STRENGTH + 52-WEEK HIGH SCANNER")
print("=" * 70)

# ── Load data: memory → CSV → fresh download ─────────────────
try:
    _ = sp500_stock_data, sp500_ticker_info, spy_close
    print("\n  ✅ Using in-memory S&P 500 data.")
except NameError:
    if os.path.exists(SP500_HIST_CSV) and os.path.exists(SPY_CSV):
        print("\n  📂 Loading S&P 500 data from CSV files...")
        df_all = pd.read_csv(SP500_HIST_CSV, parse_dates=["date"])
        sp500_ticker_info = {}; sp500_stock_data = {}
        for sym, grp in df_all.groupby("symbol"):
            yf_t = sym.replace(".", "-"); grp = grp.set_index("date").sort_index()
            sp500_stock_data[yf_t]  = grp[["open","high","low","close","volume"]]
            sp500_ticker_info[yf_t] = {"symbol": sym, "name": grp["name"].iloc[0], "sector": grp["sector"].iloc[0]}
        spy_close = pd.read_csv(SPY_CSV, index_col=0, parse_dates=True).squeeze()
        print(f"  Loaded {len(sp500_stock_data)} stocks from CSV.")
    else:
        print("\n  ⬇️  CSV not found — downloading S&P 500 data now (run S&P 500 Download cell first to avoid this)...")
        end_date   = datetime.today().strftime("%Y-%m-%d")
        start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
        html  = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={"User-Agent": "Mozilla/5.0"}).text
        sp500 = pd.read_html(io.StringIO(html), flavor="lxml")[0]
        sp500["yf_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
        sp500_ticker_info  = {row["yf_ticker"]: {"symbol": row["Symbol"], "name": row["Security"], "sector": row["GICS Sector"]} for _, row in sp500.iterrows()}
        spy_raw   = yf.download("SPY", start=start_date, end=end_date, auto_adjust=True, progress=False)
        spy_close = spy_raw["Close"].squeeze()
        all_tickers = list(sp500_ticker_info.keys())
        batches = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
        sp500_stock_data = {}
        for i, batch in enumerate(batches):
            print(f"  Batch {i+1}/{len(batches)}...", end=" ", flush=True)
            try:
                raw = yf.download(batch, start=start_date, end=end_date, auto_adjust=True, progress=False, threads=True)
                if len(batch) == 1:
                    if not raw.empty: sp500_stock_data[batch[0]] = raw.rename(columns=str.lower)
                else:
                    for t in batch:
                        try:
                            df = raw.xs(t, level=1, axis=1).dropna(how="all")
                            if not df.empty: sp500_stock_data[t] = df.rename(columns=str.lower)
                        except KeyError: pass
                print(f"ok ({sum(1 for t in batch if t in sp500_stock_data)}/{len(batch)})")
            except Exception as e: print(f"ERROR: {e}")
            time.sleep(0.3)

# ── SPY RSI ──────────────────────────────────────────────────
spy_rsi = float(_rsi(pd.Series(spy_close.values, index=spy_close.index), RSI_PERIOD).iloc[-1])
print(f"\n  SPY RSI(14) = {spy_rsi:.2f}  |  Min stock RSI = {spy_rsi+RSI_MIN_EDGE:.2f}")
print(f"  52w-high buffer = {HIGH_52W_BUFFER*100:.0f}%  |  SMA50 band = ±{SMA50_BAND*100:.0f}%")
print(f"\n  Scanning {len(sp500_stock_data)} stocks...")

# ── Scan ─────────────────────────────────────────────────────
rsi_results = []
for yf_t, df in sp500_stock_data.items():
    if len(df) < max(RSI_PERIOD+5, 50): continue
    info = sp500_ticker_info.get(yf_t, {}); close = df["close"]; last_close = float(close.iloc[-1])
    stock_rsi = float(_rsi(close, RSI_PERIOD).iloc[-1]); rsi_edge = stock_rsi - spy_rsi
    high_52w  = float(close.max()); pct_from_high = (high_52w - last_close) / high_52w
    sma50_val = close.rolling(50, min_periods=50).mean().iloc[-1]
    pct_from_sma50 = float("nan") if pd.isna(sma50_val) else abs(last_close - float(sma50_val)) / float(sma50_val)
    f_rsi = rsi_edge >= RSI_MIN_EDGE; f_52w = pct_from_high <= HIGH_52W_BUFFER
    f_sma = (not np.isnan(pct_from_sma50)) and (pct_from_sma50 <= SMA50_BAND)
    rsi_results.append({
        "symbol": info.get("symbol", yf_t), "name": info.get("name",""), "sector": info.get("sector",""),
        "confirmed": f_rsi and f_52w and f_sma,
        "rsi_14": round(stock_rsi,2), "spy_rsi_14": round(spy_rsi,2), "rsi_edge": round(rsi_edge,2),
        "pct_from_52w_high": round(pct_from_high*100,2),
        "pct_from_sma50": round(pct_from_sma50*100,2) if not np.isnan(pct_from_sma50) else None,
        "last_close": round(last_close,4), "f_rsi": f_rsi, "f_52w_high": f_52w, "f_sma50": f_sma,
    })

confirmed_rsi = [r for r in rsi_results if r["confirmed"]]
print(f"  Scan complete: {len(rsi_results)} stocks | {len(confirmed_rsi)} passed all filters\n")

print("=" * 100)
print(f"  RSI STRENGTH + 52-WEEK HIGH — {len(confirmed_rsi)} CONFIRMED")
print("=" * 100)
if confirmed_rsi:
    print(f"  {'Symbol':<8} {'Name':<28} {'Sector':<24} {'RSI':>6} {'Edge':>6} {'%Hi':>7} {'%SMA50':>8}")
    print(f"  {'-'*8} {'-'*28} {'-'*24} {'-'*6} {'-'*6} {'-'*7} {'-'*8}")
    for r in sorted(confirmed_rsi, key=lambda x: -x["rsi_edge"]):
        sma_str = f"{r['pct_from_sma50']:6.1f}%" if r["pct_from_sma50"] is not None else "   N/A"
        print(f"  {r['symbol']:<8} {r['name'][:28]:<28} {r['sector'][:24]:<24} {r['rsi_14']:>6.1f} {r['rsi_edge']:>+6.1f} {r['pct_from_52w_high']:>6.1f}% {sma_str:>8}")
else:
    print("  No stocks passed all three filters today.")
print("=" * 100)

fieldnames = ["symbol","name","sector","confirmed","rsi_14","spy_rsi_14","rsi_edge","pct_from_52w_high","pct_from_sma50","last_close","f_rsi","f_52w_high","f_sma50"]
with open(OUTPUT_RSI_CSV,"w",newline="",encoding="utf-8") as f:
    w = _csv.DictWriter(f, fieldnames=fieldnames); w.writeheader(); w.writerows(sorted(rsi_results, key=lambda r: r["confirmed"], reverse=True))
with open(OUTPUT_RSI_WL,"w",encoding="utf-8") as f:
    [f.write(f"{r['symbol']}\n") for r in confirmed_rsi]
print(f"\n  Results → {OUTPUT_RSI_CSV}   |   TV list → {OUTPUT_RSI_WL}")